In [4]:
import ee
import geopandas as gpd
import pandas as pd

In [5]:
ee.Authenticate(auth_mode="localhost")
ee.Initialize(project="wildfireproject-506722")

## 1. ERA5-Land Data Source

In [6]:
# Getting ERA5-Land data in the format of daily aggreggation

era5 = ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR")

### Choosing the variables
The initial weather feature set was kept relatively small, focusing on variables with a clear physical relationship with wildfire occurrence and vegetation/fuel dryness.

#### Selected raw variables

- **`temperature_2m`** : represents near-surface air temperature.
- **`temperature_2m_max`** : captures daily heat extremes, which may contribute to fuel drying.
- **`dewpoint_temperature_2m`** : used together with air temperature to estimate relative humidity.
- **`total_precipitation_sum`** : represents daily rainfall and will also be used to derive accumulated precipitation and dry-period indicators.
- **`volumetric_soil_water_layer_1`** : soil moisture in the upper 0–7 cm layer, used as an indicator of surface dryness.
- **`u_component_of_wind_10m`** and **`v_component_of_wind_10m`** : used to derive wind speed.
- **`potential_evaporation_sum`** : represents atmospheric/land-surface drying demand and may provide additional information beyond temperature and precipitation.
- **`surface_solar_radiation_downwards_sum`** : represents incoming solar energy and may help characterize drying conditions.

#### Not included initially

- **`leaf_area_index_high_vegetation` / `leaf_area_index_low_vegetation`** — excluded because vegetation condition will be explicitly represented using Sentinel-2 features such as NDVI and NDMI.

#### Derived features

The daily ERA5-Land variables will later be transformed into weekly and rolling weather indicators, including approximately:

- mean and maximum temperature;
- relative humidity derived from temperature and dew point;
- mean / maximum wind speed;
- precipitation accumulated over the previous 7, 14 and 30 days;
- number of consecutive dry days;
- mean surface soil moisture;
- accumulated potential evaporation;
- accumulated solar radiation.

All weather features will use only information available **before the weekly reference date**, preventing temporal leakage into the 7-day wildfire prediction window.

In [7]:
weather_variables = [
    "temperature_2m",
    "temperature_2m_max",
    "dewpoint_temperature_2m",
    "total_precipitation_sum",
    "volumetric_soil_water_layer_1",
    "u_component_of_wind_10m",
    "v_component_of_wind_10m",
    "potential_evaporation_sum",
    "surface_solar_radiation_downwards_sum"
]

#### Creating wind speed variable

In [8]:
def add_wind_speed(image):
    u = image.select("u_component_of_wind_10m")
    v = image.select("v_component_of_wind_10m")

    wind_speed = (
        u.pow(2)
        .add(v.pow(2))
        .sqrt()
        .rename("wind_speed")
    )

    return image.addBands(wind_speed)

#### Creating relative humidity variable
ERA5-Land provides air temperature and dew point temperature, but does not directly provide relative humidity in this dataset. 

Relative humidity was therefore derived using the Magnus approximation:

$$
RH = 100 \times
\exp\left(
\frac{17.625\,T_d}{243.04 + T_d}
-
\frac{17.625\,T}{243.04 + T}
\right)
$$

where:

- $RH$ = relative humidity (%)
- $T$ = 2 m air temperature (°C)
- $T_d$ = 2 m dew point temperature (°C)

Because the ERA5-Land daily product contains daily aggregated temperature and dew point values, this feature should be interpreted as an **estimated daily relative humidity**, rather than the exact daily minimum relative humidity.

In [9]:
def add_relative_humidity(image):
    # ERA5-Land temperatures are in Kelvin, so first we need to convert to Celsius
    temperature = image.select("temperature_2m").subtract(273.15)
    dewpoint = image.select("dewpoint_temperature_2m").subtract(273.15)

    # Magnus formula
    term_dewpoint = (
        dewpoint.multiply(17.625)
        .divide(dewpoint.add(243.04))
    )

    term_temperature = (
        temperature.multiply(17.625)
        .divide(temperature.add(243.04))
    )

    relative_humidity = (
        term_dewpoint
        .subtract(term_temperature)
        .exp()
        .multiply(100)
        .clamp(0, 100)
        .rename("relative_humidity")
    )

    return image.addBands(relative_humidity)

## 2. Weather Feature Engineering

The meteorological variables used in this project were selected to represent the main environmental conditions related to wildfire occurrence: 

**temperature, atmospheric dryness, precipitation, soil moisture, wind and surface drying conditions**.

#### Temperature

- `temperature_mean_7d` : mean air temperature during the previous 7 days.
- `temperature_max_7d` : highest daily maximum temperature during the previous 7 days.

The mean represents the general conditions during the previous week, while the maximum captures short periods of more extreme heat that may contribute to fuel and vegetation drying.

#### Atmospheric moisture

ERA5-Land provides air temperature and dew point temperature, which are used to estimate **relative humidity** (RH).

- `relative_humidity_mean_7d` : mean estimated relative humidity during the previous 7 days.
- `vpd_mean_7d` : mean Vapor Pressure Deficit during the previous 7 days.

Vapor Pressure Deficit (VPD) represents the difference between the amount of moisture the atmosphere can hold and the amount that is actually present. It provides an indication of the atmospheric demand for water. VPD is also used in the updated **US National Fire Danger Rating System (NFDRS v4)** as part of its fuel-moisture modelling.

Relative humidity and VPD are **related**, so their correlation will later be evaluated before fitting linear models.

#### Precipitation and antecedent dryness

Precipitation is considered over several temporal windows because recent rainfall and longer-term rainfall deficits may represent different conditions:

- `precipitation_sum_7d`
- `precipitation_sum_14d`
- `precipitation_sum_30d`

A longer precipitation history is included only for rainfall at this stage, rather than generating 14- and 30-day versions of every weather variable. This keeps the feature set relatively compact while still representing accumulated dryness.

Also:

- `consecutive_dry_days` : number of consecutive days immediately before the reference date with daily precipitation below **1 mm**.

The 1 mm threshold follows the definition used for the **Consecutive Dry Days (CDD)** climate index from the WMO/ETCCDI framework, where a dry day is defined as a day with precipitation below 1 mm.

#### Soil moisture

Surface soil moisture is represented using the first ERA5-Land soil layer (approximately 0–7 cm):

- `soil_moisture_mean_7d` : average surface soil moisture over the previous week.
- `soil_moisture_last_day` : soil moisture on the day immediately before the prediction.
- `soil_moisture_change_7d` : change between the last and first day of the 7-day window.

The change feature is calculated as:

$$
\Delta SM_{7d} = SM_{last} - SM_{first}
$$

A negative value indicates that the surface soil became drier during the week, while a positive value indicates increasing soil moisture.

#### Wind

ERA5-Land provides the horizontal wind components `u` and `v`. These components are combined into wind speed:

$$
WindSpeed = \sqrt{u^2 + v^2}
$$

The original components are not kept as model features. Instead, two weekly indicators are created:

- `wind_speed_mean_7d`
- `wind_speed_max_daily_mean_7d`

This represents both the typical wind conditions and the strongest wind conditions observed during the previous week.

#### Surface drying and energy

Three additional features are included to represent the potential drying of soil and vegetation:

- `potential_evaporation_sum_7d`
- `solar_radiation_sum_7d`
- `water_balance_7d`

Potential evaporation represents the amount of evaporation that could occur under the atmospheric and surface conditions represented by ERA5-Land, while incoming solar radiation represents the energy received at the surface.

A simple weekly water-balance proxy is also derived:

$$
WaterBalance_{7d} = Precipitation_{7d} - PotentialEvaporation_{7d}
$$

Negative values indicate that potential atmospheric water loss exceeded precipitation during the week.

#### Final weather feature set

The initial weather dataset contains the following 16 features:

1. `temperature_mean_7d`
2. `temperature_max_7d`
3. `relative_humidity_mean_7d`
4. `vpd_mean_7d`
5. `precipitation_sum_7d`
6. `precipitation_sum_14d`
7. `precipitation_sum_30d`
8. `consecutive_dry_days`
9. `soil_moisture_mean_7d`
10. `soil_moisture_last_day`
11. `soil_moisture_change_7d`
12. `wind_speed_mean_7d`
13. `wind_speed_max_daily_mean_7d`
14. `potential_evaporation_sum_7d`
15. `solar_radiation_sum_7d`
16. `water_balance_7d`

Before fitting the Logistic Regression model, correlations and multicollinearity between weather features will be examined. This is particularly relevant for combinations such as temperature and solar radiation, relative humidity and VPD, and precipitation/potential evaporation and the derived water-balance feature. Feature selection for the linear model may therefore differ from the feature set later used by non-linear models such as XGBoost.

#### Creating VPD Variable

Vapor Pressure Deficit (VPD) represents the difference between the saturation vapor pressure of the air and its actual vapor pressure. In practical terms, higher VPD values indicate a greater atmospheric demand for moisture and therefore stronger drying conditions.

VPD is calculated as:

$$
VPD = e_s(T) - e_a(T_d)
$$

where:

- $VPD$ = Vapor Pressure Deficit (kPa)
- $T$ = air temperature at 2 m (°C)
- $T_d$ = dew point temperature at 2 m (°C)
- $e_s(T)$ = saturation vapor pressure at air temperature (kPa)
- $e_a(T_d)$ = actual vapor pressure estimated from dew point temperature (kPa)

Saturation vapor pressure is estimated using:

$$
e_s(T) =
0.6108 \times
\exp\left(
\frac{17.27T}{T + 237.3}
\right)
$$

Actual vapor pressure is estimated using the same equation with dew point temperature:

$$
e_a(T_d) =
0.6108 \times
\exp\left(
\frac{17.27T_d}{T_d + 237.3}
\right)
$$

Therefore:

$$
VPD =
0.6108
\left[
\exp\left(\frac{17.27T}{T + 237.3}\right)
-
\exp\left(\frac{17.27T_d}{T_d + 237.3}\right)
\right]
$$

Since ERA5-Land stores temperature and dew point in Kelvin, both variables are first converted to Celsius:

$$
T_{^\circ C} = T_K - 273.15
$$

VPD values are expressed in **kPa**. Higher values indicate drier atmospheric conditions and a greater potential for moisture loss from vegetation and fuels.

In [10]:
def add_vpd(image):
    # Convert ERA5-Land temperature from Kelvin to Celsius
    temperature = image.select("temperature_2m").subtract(273.15)
    dewpoint = image.select("dewpoint_temperature_2m").subtract(273.15)

    # Saturation vapour pressure (kPa)
    saturation_vapour_pressure = (
        temperature.multiply(17.27)
        .divide(temperature.add(237.3))
        .exp()
        .multiply(0.6108)
    )

    # Actual vapour pressure estimated from dew point (kPa)
    actual_vapour_pressure = (
        dewpoint.multiply(17.27)
        .divide(dewpoint.add(237.3))
        .exp()
        .multiply(0.6108)
    )

    # Vapor Pressure Deficit
    vpd = (
        saturation_vapour_pressure
        .subtract(actual_vapour_pressure)
        .max(0)
        .rename("vpd")
    )

    return image.addBands(vpd)

## 3. Weekly Weather Feature Construction

The functions below automate the construction of the weather features for any weekly reference date.

For a reference date on Monday, all meteorological variables are calculated using observations strictly before that date. The 7-day window therefore corresponds to the previous Monday–Sunday period, while precipitation also uses 14- and 30-day antecedent windows.

The final weekly ERA5-Land image contains the 16 weather features defined above.

In [11]:
def build_weather_windows(reference_date):
    reference_date = ee.Date(reference_date)

    start_7d = reference_date.advance(-7, "day")
    start_14d = reference_date.advance(-14, "day")
    start_30d = reference_date.advance(-30, "day")

    weather_7d = era5.filterDate(start_7d, reference_date)
    weather_14d = era5.filterDate(start_14d, reference_date)
    weather_30d = era5.filterDate(start_30d, reference_date)

    return weather_7d, weather_14d, weather_30d

In [12]:
def build_temperature_moisture_features(weather_7d_features):

    temperature_mean_7d = (
        weather_7d_features
        .select("temperature_2m")
        .mean()
        .subtract(273.15)
        .rename("temperature_mean_7d")
    )

    temperature_max_7d = (
        weather_7d_features
        .select("temperature_2m_max")
        .max()
        .subtract(273.15)
        .rename("temperature_max_7d")
    )

    relative_humidity_mean_7d = (
        weather_7d_features
        .select("relative_humidity")
        .mean()
        .rename("relative_humidity_mean_7d")
    )

    vpd_mean_7d = (
        weather_7d_features
        .select("vpd")
        .mean()
        .rename("vpd_mean_7d")
    )

    return (
        temperature_mean_7d
        .addBands(temperature_max_7d)
        .addBands(relative_humidity_mean_7d)
        .addBands(vpd_mean_7d)
    )

In [13]:
def build_precipitation_features(
    weather_7d_features,
    weather_14d,
    weather_30d
):

    precipitation_sum_7d = (
        weather_7d_features
        .select("total_precipitation_sum")
        .sum()
        .multiply(1000)  # m -> mm
        .rename("precipitation_sum_7d")
    )

    precipitation_sum_14d = (
        weather_14d
        .select("total_precipitation_sum")
        .sum()
        .multiply(1000)
        .rename("precipitation_sum_14d")
    )

    precipitation_sum_30d = (
        weather_30d
        .select("total_precipitation_sum")
        .sum()
        .multiply(1000)
        .rename("precipitation_sum_30d")
    )

    return (
        precipitation_sum_7d
        .addBands(precipitation_sum_14d)
        .addBands(precipitation_sum_30d)
    )

In [14]:
def build_soil_moisture_features(weather_7d_features):

    weather_7d_sorted = weather_7d_features.sort("system:time_start")

    first_day = weather_7d_sorted.first()

    last_day = (
        weather_7d_sorted
        .sort("system:time_start", False)
        .first()
    )

    soil_moisture_mean_7d = (
        weather_7d_features
        .select("volumetric_soil_water_layer_1")
        .mean()
        .rename("soil_moisture_mean_7d")
    )

    soil_moisture_last_day = (
        last_day
        .select("volumetric_soil_water_layer_1")
        .rename("soil_moisture_last_day")
    )

    soil_moisture_change_7d = (
        last_day
        .select("volumetric_soil_water_layer_1")
        .subtract(
            first_day.select("volumetric_soil_water_layer_1")
        )
        .rename("soil_moisture_change_7d")
    )

    return (
        soil_moisture_mean_7d
        .addBands(soil_moisture_last_day)
        .addBands(soil_moisture_change_7d)
    )

In [15]:
def build_wind_features(weather_7d_features):

    wind_speed_mean_7d = (
        weather_7d_features
        .select("wind_speed")
        .mean()
        .rename("wind_speed_mean_7d")
    )

    wind_speed_max_daily_mean_7d = (
        weather_7d_features
        .select("wind_speed")
        .max()
        .rename("wind_speed_max_daily_mean_7d")
    )

    return (
        wind_speed_mean_7d
        .addBands(wind_speed_max_daily_mean_7d)
    )

In [16]:
def build_drying_energy_features(weather_7d_features):

    potential_evaporation_sum_7d = (
        weather_7d_features
        .select("potential_evaporation_sum")
        .sum()
        .multiply(-1000)  # m -> mm and invert ERA5-Land sign convention
        .rename("potential_evaporation_sum_7d")
    )

    solar_radiation_sum_7d = (
        weather_7d_features
        .select("surface_solar_radiation_downwards_sum")
        .sum()
        .divide(1_000_000)  # J/m² -> MJ/m²
        .rename("solar_radiation_sum_7d")
    )

    precipitation_sum_7d = (
        weather_7d_features
        .select("total_precipitation_sum")
        .sum()
        .multiply(1000)  # m -> mm
    )

    water_balance_7d = (
        precipitation_sum_7d
        .subtract(potential_evaporation_sum_7d)
        .rename("water_balance_7d")
    )

    return (
        potential_evaporation_sum_7d
        .addBands(solar_radiation_sum_7d)
        .addBands(water_balance_7d)
    )

In [17]:
def build_consecutive_dry_days_image(reference_date, lookback_days=365):

    reference_date = ee.Date(reference_date)

    start_date = reference_date.advance(
        -lookback_days,
        "day"
    )

    precipitation = (
        era5
        .filterDate(start_date, reference_date)
        .select("total_precipitation_sum")
    )

    def wet_day_date(image):

        wet_day = image.gte(0.001)

        day_index = (
            ee.Image.constant(
                ee.Date(
                    image.get("system:time_start")
                ).difference(
                    ee.Date("1970-01-01"),
                    "day"
                )
            )
            .toInt32()
        )

        return (
            day_index
            .updateMask(wet_day)
            .rename("wet_day_index")
        )

    wet_day_dates = precipitation.map(wet_day_date)

    last_wet_day = wet_day_dates.max()

    reference_day_index = (
        ee.Image.constant(
            reference_date.difference(
                ee.Date("1970-01-01"),
                "day"
            )
        )
        .toInt32()
    )

    consecutive_dry_days = (
        reference_day_index
        .subtract(last_wet_day)
        .subtract(1)
        .rename("consecutive_dry_days")
    )

    return consecutive_dry_days

In [18]:
def build_weekly_weather_image(reference_date):

    weather_7d, weather_14d, weather_30d = build_weather_windows(
        reference_date
    )

    weather_7d_features = (
        weather_7d
        .select(weather_variables)
        .map(add_wind_speed)
        .map(add_relative_humidity)
        .map(add_vpd)
    )

    temperature_moisture = build_temperature_moisture_features(
        weather_7d_features
    )

    precipitation = build_precipitation_features(
        weather_7d_features,
        weather_14d,
        weather_30d
    )

    soil_moisture = build_soil_moisture_features(
        weather_7d_features
    )

    wind = build_wind_features(
        weather_7d_features
    )

    drying_energy = build_drying_energy_features(
        weather_7d_features
    )

    consecutive_dry_days = build_consecutive_dry_days_image(
        reference_date
    )

    weekly_features_image = (
        temperature_moisture
        .addBands(precipitation)
        .addBands(soil_moisture)
        .addBands(wind)
        .addBands(drying_energy)
        .addBands(consecutive_dry_days)
    )

    return weekly_features_image

In [19]:
test_image = build_weekly_weather_image("2018-08-20")

print(test_image.bandNames().size().getInfo())
print(test_image.bandNames().getInfo())

16
['temperature_mean_7d', 'temperature_max_7d', 'relative_humidity_mean_7d', 'vpd_mean_7d', 'precipitation_sum_7d', 'precipitation_sum_14d', 'precipitation_sum_30d', 'soil_moisture_mean_7d', 'soil_moisture_last_day', 'soil_moisture_change_7d', 'wind_speed_mean_7d', 'wind_speed_max_daily_mean_7d', 'potential_evaporation_sum_7d', 'solar_radiation_sum_7d', 'water_balance_7d', 'consecutive_dry_days']


## 4. Spatial Matching Between the 5 km Grid and ERA5-Land

#### Problem

The ICNF dataset uses a regular **5 × 5 km grid**, while ERA5-Land uses **11 km**. Because of this difference, several 5 km cells may naturally share the same ERA5-Land pixel.

An additional problem appeared for coastal cells. When weather values were extracted directly using `reduceRegions()`, some cells returned `None`. This happened because ERA5-Land is a land-only product and pixels classified as ocean are masked. Therefore, a coastal 5 km cell can contain land while its centre is located on the ocean.

For example, `PT_0002` contains approximately **52% land**, but direct extraction returned no ERA5-Land values.

#### Possible solutions

Several alternatives were considered:

- **Average ERA5-Land values inside each 5 km polygon.**  
  This worked for many inland cells but failed for some coastal cells because no valid ERA5-Land pixel was available inside the polygon.

- **Use the geometric centroid of each 5 km cell.**  
  This is simple, but for coastal cells the centroid can fall over the ocean and therefore return no ERA5-Land value.

- **Use a buffer around the cell and calculate the mean weather conditions.**  
  This solves the missing-value problem, but introduces an arbitrary spatial averaging area and may mix conditions from locations considerably farther away from the actual cell.

- **Associate each cell with the nearest valid ERA5-Land pixel.**  
  This respects the true spatial resolution of ERA5-Land and avoids pretending that meteorological information is available at 5 km resolution.

The final option was considered the most appropriate.

#### Final solution

For each 5 km cell, only the portion intersecting Mainland Portugal was considered. A `representative_point()` was then generated from this land portion, ensuring that the reference point is located on land even for coastal cells.

Valid ERA5-Land pixel centres were extracted from Google Earth Engine using a **25 km buffer around Mainland Portugal**. The buffer was included so that cells close to the Spanish border could also be matched to nearby ERA5-Land pixels located outside the Portuguese border.

The ERA5-Land pixels and grid representative points were converted to the same projected coordinate system (**EPSG:3763 — ETRS89 / Portugal TM06**), allowing distances to be calculated in metres.

Each grid cell was then associated with its nearest valid ERA5-Land pixel using a nearest spatial join:

`cell_id -> representative land point -> nearest valid ERA5-Land pixel`

ERA5-Land pixel identifiers were created from their geographic coordinates instead of a sequential index, making the identifiers reproducible if the pixel collection is generated again.

#### Results

A total of **1,172 valid ERA5-Land pixels** were identified within the 25 km study-area buffer.

All **3,822 grid cells** were successfully matched to an ERA5-Land pixel, with **no missing mappings**.

The distance distribution between each grid cell's representative land point and its assigned ERA5-Land pixel was:

- Mean: **4.04 km**
- Median: **3.99 km**
- 90th percentile: **5.95 km**
- 95th percentile: **6.53 km**
- 99th percentile: **11.17 km**

Only **59 cells (1.54%)** were more than 10 km from their assigned ERA5-Land pixel.

Among these, only **19 cells (0.50% of the complete grid)** had at least 50% of their area on land. For these mostly-land cells, the maximum matching distance was **12.82 km**, which is close to the native spatial scale of ERA5-Land.

The largest distance observed was **28.98 km**, but this occurred in a highly marginal coastal cell (`PT_0006`) containing only approximately **3.2% land**.

No cells were removed based on distance at this stage. Instead, both `land_fraction` and `distance_km` are retained so that these cases remain identifiable and can be investigated in a later sensitivity analysis if necessary.

Finally, although the wildfire grid contains **3,822 cells**, only **962 unique ERA5-Land pixels** are actually required to represent their meteorological conditions. This avoids calculating identical weather features repeatedly for neighbouring 5 km cells that share the same ERA5-Land pixel and substantially reduces the amount of weather data that needs to be processed.

In [20]:
districts = gpd.read_file(
    "../data/raw/CAOP_Continente_2025-gpkg/Continente_CAOP2025.gpkg",
    layer="cont_distritos"
)

portugal = districts.geometry.union_all()

In [21]:
portugal_buffer = portugal.buffer(25000)

portugal_buffer_wgs84 = (
    gpd.GeoSeries(
        [portugal_buffer],
        crs=districts.crs
    )
    .to_crs(epsg=4326)
    .iloc[0]
)

portugal_buffer_ee = ee.Geometry(
    portugal_buffer_wgs84.__geo_interface__
)

In [22]:
era5_pixels_buffer = (
    test_image
    .select("temperature_mean_7d")
    .sample(
        region=portugal_buffer_ee,
        scale=11132,
        geometries=True,
        dropNulls=True
    )
)

print(
    "Valid ERA5-Land pixels around Mainland Portugal:",
    era5_pixels_buffer.size().getInfo()
)

Valid ERA5-Land pixels around Mainland Portugal: 1172


In [23]:
grid_5km_portugal = gpd.read_parquet(
    "../data/interim/grid_5km_portugal.parquet"
)

print(grid_5km_portugal.shape)
print(grid_5km_portugal.crs)

(3822, 4)
{"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "ProjectedCRS", "name": "ETRS89 / Portugal TM06", "base_crs": {"name": "ETRS89", "datum_ensemble": {"name": "European Terrestrial Reference System 1989 ensemble", "members": [{"name": "European Terrestrial Reference Frame 1989"}, {"name": "European Terrestrial Reference Frame 1990"}, {"name": "European Terrestrial Reference Frame 1991"}, {"name": "European Terrestrial Reference Frame 1992"}, {"name": "European Terrestrial Reference Frame 1993"}, {"name": "European Terrestrial Reference Frame 1994"}, {"name": "European Terrestrial Reference Frame 1996"}, {"name": "European Terrestrial Reference Frame 1997"}, {"name": "European Terrestrial Reference Frame 2000"}, {"name": "European Terrestrial Reference Frame 2005"}, {"name": "European Terrestrial Reference Frame 2014"}, {"name": "European Terrestrial Reference Frame 2020"}], "ellipsoid": {"name": "GRS 1980", "semi_major_axis": 6378137, "inverse_flattenin

In [24]:
era5_buffer_info = era5_pixels_buffer.getInfo()

pixel_records = []

for feature in era5_buffer_info["features"]:
    longitude, latitude = feature["geometry"]["coordinates"]

    pixel_records.append({
        "era5_pixel_id": f"ERA5_{longitude:.6f}_{latitude:.6f}",
        "longitude": longitude,
        "latitude": latitude
    })

era5_pixels_gdf = gpd.GeoDataFrame(
    pixel_records,
    geometry=gpd.points_from_xy(
        [row["longitude"] for row in pixel_records],
        [row["latitude"] for row in pixel_records]
    ),
    crs="EPSG:4326"
).to_crs(grid_5km_portugal.crs)

In [25]:
cell_land_points = grid_5km_portugal[
    ["cell_id", "land_fraction", "geometry"]
].copy()

# Keep only the part of each cell that overlaps Mainland Portugal
cell_land_points["geometry"] = (
    cell_land_points.geometry
    .intersection(portugal)
    .representative_point()
)

In [26]:
cell_era5_mapping = gpd.sjoin_nearest(
    cell_land_points,
    era5_pixels_gdf[
        ["era5_pixel_id", "longitude", "latitude", "geometry"]
    ],
    how="left",
    distance_col="distance_m"
)

cell_era5_mapping["distance_km"] = (
    cell_era5_mapping["distance_m"] / 1000
)

In [27]:
far_mostly_land = cell_era5_mapping[
    (cell_era5_mapping["distance_km"] > 10) &
    (cell_era5_mapping["land_fraction"] >= 0.5)
].copy()

print(
    "Cells > 10 km and >= 50% land:",
    len(far_mostly_land)
)

print(
    far_mostly_land["distance_km"].describe()
)

far_mostly_land[
    ["cell_id", "land_fraction", "era5_pixel_id", "distance_km"]
].sort_values(
    "distance_km",
    ascending=False
).head(20)

Cells > 10 km and >= 50% land: 19
count    19.000000
mean     11.306096
std       0.777931
min      10.002943
25%      10.647954
50%      11.235332
75%      11.811768
max      12.819819
Name: distance_km, dtype: float64


,cell_id,land_fraction,era5_pixel_id,distance_km
48,PT_0049,0.826972,ERA5_-9.250042_39.250180,12.819819
279,PT_0280,0.885980,ERA5_-8.750040_37.250170,12.352435
228,PT_0229,0.725657,ERA5_-8.750040_37.250170,12.121009
94,PT_0095,0.723973,ERA5_-9.050041_39.450180,12.083574
373,PT_0374,1.000000,ERA5_-8.650040_38.450176,11.901893
227,PT_0228,0.945861,ERA5_-8.750040_37.150170,11.721642
372,PT_0373,0.872033,ERA5_-8.650040_38.350175,11.637007
618,PT_0619,0.686934,ERA5_-8.550039_40.850187,11.556124
617,PT_0618,0.939083,ERA5_-8.550039_40.750186,11.501211
147,PT_0148,0.910453,ERA5_-9.050041_39.450180,11.235332


In [28]:
cell_era5_mapping.to_parquet(
    "../data/interim/cell_era5_mapping.parquet",
    index=False
)

In [29]:
used_pixel_ids = (
    cell_era5_mapping["era5_pixel_id"]
    .unique()
)

era5_pixels_used = (
    era5_pixels_gdf[
        era5_pixels_gdf["era5_pixel_id"].isin(used_pixel_ids)
    ]
    .copy()
)

print(era5_pixels_used.shape)

(962, 4)


In [30]:
era5_features = []

for _, row in era5_pixels_used.iterrows():

    geometry = ee.Geometry.Point([
        row["longitude"],
        row["latitude"]
    ])

    feature = ee.Feature(
        geometry,
        {"era5_pixel_id": row["era5_pixel_id"]}
    )

    era5_features.append(feature)

era5_pixels_ee = ee.FeatureCollection(era5_features)

print(
    "ERA5 pixels in GEE:",
    era5_pixels_ee.size().getInfo()
)

ERA5 pixels in GEE: 962


## 5. Annual Weather Extraction

Extracting every cell-week independently would require a large number of repeated Earth Engine operations. Since the 3,822 grid cells are represented by only **962 unique ERA5-Land pixels**, weather features are calculated only for these unique pixels.

The 229 reference weeks are processed by year. For each year, the weekly 16-band images are combined into a single wide image and sampled once at the ERA5-Land pixel locations. This avoids the concurrent-aggregation limitations encountered when sampling every week separately.

The yearly results are exported as CSV files and subsequently reshaped into a long `ERA5 pixel × reference week` dataset.

In [31]:
model_dataset = pd.read_parquet(
    "../data/interim/model_dataset_weekly.parquet"
)

reference_dates = (
    model_dataset["reference_date"]
    .drop_duplicates()
    .sort_values()
)

YEARS = sorted(
    reference_dates.dt.year.unique().tolist()
)

N_ERA5_PIXELS = (
    era5_pixels_used["era5_pixel_id"]
    .nunique()
)

print("Years:", YEARS)
print("ERA5 pixels:", N_ERA5_PIXELS)

print("Reference dates:", len(reference_dates))
print(reference_dates.min())
print(reference_dates.max())

print(
    reference_dates
    .dt.year
    .value_counts()
    .sort_index()
)

Years: [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
ERA5 pixels: 962
Reference dates: 229
2017-05-01 00:00:00
2025-10-20 00:00:00
reference_date
2017    26
2018    25
2019    25
2020    25
2021    26
2022    26
2023    26
2024    25
2025    25
Name: count, dtype: int64


In [32]:
def build_yearly_weather_image(year, reference_dates):

    year_dates = (
        reference_dates[
            reference_dates.dt.year == year
        ]
        .dt.strftime("%Y-%m-%d")
        .tolist()
    )

    yearly_image = None

    for date in year_dates:

        weekly_image = build_weekly_weather_image(date)

        date_prefix = date.replace("-", "")

        new_band_names = weekly_image.bandNames().map(
            lambda band: (
                ee.String(date_prefix)
                .cat("__")
                .cat(ee.String(band))
            )
        )

        weekly_image = weekly_image.rename(
            new_band_names
        )

        if yearly_image is None:
            yearly_image = weekly_image
        else:
            yearly_image = yearly_image.addBands(
                weekly_image
            )

    return yearly_image

In [33]:
def export_yearly_weather(year, reference_dates):

    yearly_image = build_yearly_weather_image(
        year,
        reference_dates
    )

    yearly_sample = yearly_image.sampleRegions(
        collection=era5_pixels_ee,
        properties=["era5_pixel_id"],
        scale=11132,
        geometries=False
    )

    task = ee.batch.Export.table.toDrive(
        collection=yearly_sample,
        description=f"era5_weather_{year}",
        folder="wildfire_risk_portugal",
        fileNamePrefix=f"era5_weather_{year}",
        fileFormat="CSV"
    )

    task.start()

    return task

In [34]:
def wide_weather_to_long(weather_df):

    weather_columns = [
        col for col in weather_df.columns
        if "__" in col
    ]

    weather_long = (
        weather_df[
            ["era5_pixel_id"] + weather_columns
        ]
        .melt(
            id_vars="era5_pixel_id",
            var_name="date_feature",
            value_name="value"
        )
    )

    weather_long[
        ["reference_date", "feature"]
    ] = weather_long["date_feature"].str.split(
        "__",
        n=1,
        expand=True
    )

    weather_long["reference_date"] = pd.to_datetime(
        weather_long["reference_date"],
        format="%Y%m%d"
    )

    weather_long = (
        weather_long
        .pivot(
            index=["era5_pixel_id", "reference_date"],
            columns="feature",
            values="value"
        )
        .reset_index()
    )

    weather_long.columns.name = None

    return weather_long

In [36]:
weather_tasks = {}

for year in YEARS:

    task = export_yearly_weather(
        year,
        reference_dates
    )

    weather_tasks[year] = task

    print(
        year,
        task.status()["state"]
    )

2017 READY
2018 READY
2019 READY
2020 READY
2021 READY
2022 READY
2023 READY
2024 READY
2025 READY


In [37]:
for year, task in weather_tasks.items():

    print(
        year,
        task.status()["state"]
    )

2017 COMPLETED
2018 COMPLETED
2019 COMPLETED
2020 COMPLETED
2021 COMPLETED
2022 COMPLETED
2023 COMPLETED
2024 RUNNING
2025 RUNNING


## 6. Processing and Validating the Annual Exports

The yearly exports are converted from wide to long format so that each row represents one ERA5-Land pixel during one reference week.

Each year is validated for the expected number of observations, unique pixels and weeks, duplicate pixel-week combinations, and missing values before the annual datasets are combined.

In [38]:
all_weather_years = []

for year in YEARS:

    file_path = f"../data/interim/era5_weather_{year}.csv"

    weather_wide = pd.read_csv(file_path)

    weather_long = wide_weather_to_long(
        weather_wide
    )

    expected_weeks = (
        reference_dates.dt.year
        .eq(year)
        .sum()
    )

    expected_rows = expected_weeks * N_ERA5_PIXELS

    duplicates = (
        weather_long[
            ["era5_pixel_id", "reference_date"]
        ]
        .duplicated()
        .sum()
    )

    missing = weather_long.isna().sum().sum()

    print(
        year,
        "| weeks:", expected_weeks,
        "| rows:", len(weather_long),
        "| expected:", expected_rows,
        "| duplicates:", duplicates,
        "| missing:", missing
    )

    # Stop immediately if something is wrong
    assert len(weather_long) == expected_rows
    assert (
    weather_long["era5_pixel_id"].nunique()
    == N_ERA5_PIXELS)
    assert weather_long["reference_date"].nunique() == expected_weeks
    assert duplicates == 0
    assert missing == 0

    weather_long.to_parquet(
        f"../data/interim/era5_weather_{year}.parquet",
        index=False
    )

    all_weather_years.append(
        weather_long
    )

2017 | weeks: 26 | rows: 25012 | expected: 25012 | duplicates: 0 | missing: 0
2018 | weeks: 25 | rows: 24050 | expected: 24050 | duplicates: 0 | missing: 0
2019 | weeks: 25 | rows: 24050 | expected: 24050 | duplicates: 0 | missing: 0
2020 | weeks: 25 | rows: 24050 | expected: 24050 | duplicates: 0 | missing: 0
2021 | weeks: 26 | rows: 25012 | expected: 25012 | duplicates: 0 | missing: 0
2022 | weeks: 26 | rows: 25012 | expected: 25012 | duplicates: 0 | missing: 0
2023 | weeks: 26 | rows: 25012 | expected: 25012 | duplicates: 0 | missing: 0
2024 | weeks: 25 | rows: 24050 | expected: 24050 | duplicates: 0 | missing: 0
2025 | weeks: 25 | rows: 24050 | expected: 24050 | duplicates: 0 | missing: 0


In [39]:
era5_weather_weekly = (
    pd.concat(
        all_weather_years,
        ignore_index=True
    )
    .sort_values(
        ["reference_date", "era5_pixel_id"]
    )
    .reset_index(drop=True)
)

print("Final shape:", era5_weather_weekly.shape)

print(
    "Unique pixels:",
    era5_weather_weekly["era5_pixel_id"].nunique()
)

print(
    "Unique weeks:",
    era5_weather_weekly["reference_date"].nunique()
)

print(
    "Duplicate pixel-week pairs:",
    era5_weather_weekly[
        ["era5_pixel_id", "reference_date"]
    ].duplicated().sum()
)

print(
    "Missing values:",
    era5_weather_weekly.isna().sum().sum()
)

Final shape: (220298, 18)
Unique pixels: 962
Unique weeks: 229
Duplicate pixel-week pairs: 0
Missing values: 0


In [40]:
era5_weather_weekly.to_parquet(
    "../data/interim/era5_weather_weekly.parquet",
    index=False
)

## 7. Mapping Weather Back to the 5 km Grid

The validated ERA5-Land dataset contains one observation per weather pixel and week. The previously created spatial mapping is now used to assign these weather conditions back to the 3,822 wildfire grid cells.

Neighbouring grid cells may share the same meteorological values when they are associated with the same ERA5-Land pixel. The original `era5_pixel_id` and matching distance are retained for traceability.

In [41]:
cell_weather_mapping = cell_era5_mapping[
    ["cell_id", "era5_pixel_id", "distance_km"]
].copy()

weather_grid_weekly = cell_weather_mapping.merge(
    era5_weather_weekly,
    on="era5_pixel_id",
    how="left"
)

In [42]:
print("Shape:", weather_grid_weekly.shape)

print(
    "Unique cells:",
    weather_grid_weekly["cell_id"].nunique()
)

print(
    "Unique weeks:",
    weather_grid_weekly["reference_date"].nunique()
)

print(
    "Duplicate cell-week pairs:",
    weather_grid_weekly[
        ["cell_id", "reference_date"]
    ].duplicated().sum()
)

print(
    "Missing weather values:",
    weather_grid_weekly.isna().sum().sum()
)

Shape: (875238, 20)
Unique cells: 3822
Unique weeks: 229
Duplicate cell-week pairs: 0
Missing weather values: 0


In [43]:
weather_grid_weekly.to_parquet(
    "../data/interim/weather_grid_weekly.parquet",
    index=False
)

## 8. Merging Weather Features with the Wildfire Dataset

Finally, the weekly weather features are joined to the wildfire modelling dataset using `cell_id` and `reference_date`.

Each resulting row represents one **5 × 5 km cell during one prediction week**, containing meteorological information from before the reference date and the wildfire target for the following seven days.

In [44]:
model_dataset_weather = model_dataset.merge(
    weather_grid_weekly,
    on=["cell_id", "reference_date"],
    how="left",
    validate="one_to_one"
)

In [45]:
print("Shape:", model_dataset_weather.shape)

print(
    "Unique cells:",
    model_dataset_weather["cell_id"].nunique()
)

print(
    "Unique weeks:",
    model_dataset_weather["reference_date"].nunique()
)

print(
    "Duplicate cell-week pairs:",
    model_dataset_weather[
        ["cell_id", "reference_date"]
    ].duplicated().sum()
)

weather_features = [
    col for col in era5_weather_weekly.columns
    if col not in ["era5_pixel_id", "reference_date"]
]

print(
    "Missing weather values:",
    model_dataset_weather[weather_features]
    .isna()
    .sum()
    .sum()
)

print(
    "\nTarget distribution:"
)

print(
    model_dataset_weather["fire_next_7_days"]
    .value_counts()
)

Shape: (875238, 25)
Unique cells: 3822
Unique weeks: 229
Duplicate cell-week pairs: 0
Missing weather values: 0

Target distribution:
fire_next_7_days
0    825098
1     50140
Name: count, dtype: int64


### Final Output

The resulting dataset contains **875,238 cell-week observations**, with complete weather information for all **3,822 grid cells** and **229 reference weeks** from 2017 to 2025.

The dataset is saved as:

`data/interim/model_dataset_weather.parquet`

This dataset will later be enriched with Sentinel-2 vegetation features.

In [46]:
model_dataset_weather.to_parquet(
    "../data/interim/model_dataset_weather.parquet",
    index=False
)